In [2]:
import pandas as pd 
import numpy as np

In [10]:
lpa_metrics = pd.read_csv("./Utrecht_LPA_Validation_Metrics.csv")
lpa_metrics

lga_metrics = pd.read_csv("./Utrecht_LGA_Validation_Metrics.csv")
lga_3d_metrics = pd.read_csv("./UTRECHT_LGA_Morphometrics.csv")

In [9]:
print(lpa_metrics.describe())
print(lga_metrics.describe())

            Dice        IoU       HD95  Precision     Recall
count  19.000000  19.000000  19.000000  19.000000  19.000000
mean    0.652588   0.514377  11.590215   0.643064   0.751551
std     0.201466   0.214124   8.214623   0.239567   0.194791
min     0.200341   0.111321   1.414214   0.112947   0.324176
25%     0.477658   0.314282   4.345208   0.523147   0.657194
50%     0.684799   0.520680  11.368801   0.666667   0.820868
75%     0.829844   0.709211  17.233829   0.829021   0.888809
max     0.896352   0.812171  29.778198   0.959311   0.971406
            Dice        IoU       HD95  Precision     Recall
count  19.000000  19.000000  19.000000  19.000000  19.000000
mean    0.592628   0.452549  15.654031   0.730623   0.525660
std     0.221031   0.213695  10.335564   0.231737   0.214094
min     0.138381   0.074334   3.605551   0.248750   0.082298
25%     0.469029   0.306743   6.484213   0.573461   0.433212
50%     0.668920   0.502539  12.083046   0.843345   0.550310
75%     0.768924   0.624

In [13]:
print(lga_3d_metrics)

   Subject    Vol_mm3  Area_mm2  Sphericity  Fractal_Dim    SAVR  Compactness  \
0   UTR_01   58951.04  30558.71      0.2397       1.8773  0.5184       0.0265   
1   UTR_02   63049.25  31231.27      0.2453       1.8902  0.4953       0.0314   
2   UTR_03   48875.50  23461.40      0.2755       1.9366  0.4800       0.0438   
3   UTR_04    1108.92   1333.06      0.3887       1.3557  1.2021       0.0022   
4   UTR_05    1231.96   1366.08      0.4068       1.4353  1.1089       0.0044   
5   UTR_06    6099.46   4239.45      0.3808       1.5467  0.6951       0.0053   
6   UTR_07   49282.08  21522.76      0.3020       1.8860  0.4367       0.0337   
7   UTR_08   43571.58  24546.27      0.2440       1.8952  0.5634       0.0273   
8   UTR_09    3272.67   3429.04      0.3109       1.5698  1.0478       0.0061   
9   UTR_10   50328.21  25017.34      0.2635       1.8680  0.4971       0.0295   
10  UTR_11    1981.50   2142.39      0.3561       1.4828  1.0812       0.0042   
11  UTR_12  101901.88  40780

In [25]:
from bs4 import BeautifulSoup
import os
import pandas as pd

main_folder = "/Users/alert/Downloads/WMH_EXPERIMENT_UTRECHT/SPM_UTRECHT/T1_FLAIR"

data = []

for subject_folder in os.listdir(main_folder):

    subject_path = os.path.join(main_folder, subject_folder)

    if os.path.isdir(subject_path):

        for file in os.listdir(subject_path):

            if file.endswith(".html"):

                file_path = os.path.join(subject_path, file)

                with open(file_path, "r", encoding="utf-8") as f:
                    soup = BeautifulSoup(f, "html.parser")

                lesion_volume = None
                lesion_number = None

                rows = soup.find_all("tr")

                for row in rows:
                    cols = row.find_all("td")

                    if len(cols) == 2:
                        label = cols[0].text.strip()
                        value = cols[1].text.strip()

                        if "Lesion volume" in label:
                            lesion_volume = value.replace(" ml","")

                        if "Number of lesions" in label:
                            lesion_number = value

                data.append({
                    "Subject": subject_folder,
                    "Lesion Volume (ml)": float(lesion_volume) if lesion_volume else None,
                    "Number of Lesions": int(lesion_number) if lesion_number else None
                })

lga_df = pd.DataFrame(data)

print(lga_df)

   Subject  Lesion Volume (ml)  Number of Lesions
0   UTR_15            41.28960                 29
1   UTR_12            40.84600                 15
2   UTR_13             2.35850                 11
3   UTR_14             2.55680                 12
4   UTR_09             0.95606                  9
5   UTR_07            19.59780                  8
6   UTR_01            22.05270                 20
7   UTR_06             2.17110                 11
8   UTR_08            16.31910                 27
9   UTR_11             0.57308                  8
10  UTR_16            38.93940                 11
11  UTR_18             1.46850                  8
12  UTR_19            35.50090                 21
13  UTR_17            27.92410                 11
14  UTR_10            19.36080                 21
15  UTR_03            18.99440                 18
16  UTR_04             0.30307                  6
17  UTR_05             0.31685                  4
18  UTR_02            24.44970                 34


In [24]:
lpa_clinical_metrics = pd.merge(lpa_metrics, lpa_df, on="Subject", how="inner")
lpa_clinical_metrics.to_csv("LPA_CLINICAL_METRICS.csv")

In [21]:
lga_clinical_metrics = pd.merge(lga_metrics, lga_df, on="Subject", how="inner")
lga_clinical_metrics.to_csv("LGA_CLINICAL_METRICS.csv")

In [26]:
lga_morphometrics = pd.merge(lga_3d_metrics, lga_df, on="Subject", how="inner")
lga_morphometrics.to_csv("LGA_MORPHOMETRICS.csv")